## Demo notebook

In this demo we will go through the basic functionalities of `RotOptSynth`.

In [1]:
import numpy as np
from scipy.stats import unitary_group
import rotoptsynth as ros
import pennylane as qml

In [2]:
# Pick some system size and generate a target with that size.
n = 4
n = 6
N = 2**n
wires = range(n)
dim = 4**n
U = unitary_group.rvs(N, random_state=81512)

In [9]:
# Perform unitary synthesis with validation enabled.
ros.disable_validation()
%prun -s cumtime ops = ros.rot_opt_synth(U, wires=wires, zeroed_wires=None)

# Additionally check manually that the matrix is reproduced correctly:
print(f"Matrix reproduced correctly: {np.allclose(ros.ops_to_mat(ops, wires), U)}")

 Matrix reproduced correctly: True


         2188268 function calls (2162697 primitive calls) in 0.880 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.000    0.000    0.577    0.577 {built-in method builtins.exec}
      2/1    0.000    0.000    0.577    0.577 <string>:1(<module>)
      5/1    0.304    0.061    0.576    0.576 full_decomp.py:96(rot_opt_synth)
      4/1    0.000    0.000    0.574    0.574 full_decomp.py:22(_decompose_first_mplx)
    84/12    0.003    0.000    0.565    0.047 diag_decomps.py:198(diag_decomp)
      255    0.002    0.000    0.483    0.002 diag_decomps.py:23(_diag_decomp_two_qubits)
        2    0.000    0.000    0.304    0.152 events.py:86(_run)
        2    0.000    0.000    0.304    0.152 {method 'run' of '_contextvars.Context' objects}
        1    0.000    0.000    0.304    0.304 asyncio.py:206(_handle_events)
        2    0.000    0.000    0.303    0.152 zmqstream.py:573(_handle_events)
        2    0.000    0.

In [4]:
# Count the number of rotation angles in the operators and compare to the group dimension
num_rotation_angles = ros.count_rotation_angles(ops)
print(f"Number of rotation angles ({num_rotation_angles}) matches group dimension ({dim}): {num_rotation_angles==dim}")

Number of rotation angles (16) matches group dimension (16): True


In [5]:
# Draw the circuit
print(qml.drawer.tape_text(qml.tape.QuantumScript(ops), show_matrices=False, max_length=200, wire_order=wires))

0: ──U(M1)─╭●──RY─╭X──RY─╭●──U(M3)─╭GlobalPhase─┤  
1: ──U(M0)─╰X──RZ─╰●─────╰X──U(M2)─╰GlobalPhase─┤  


In [26]:
# Count gates:
@qml.specs
@qml.qnode(qml.device("default.qubit"))
def node():
    for op in ops:
        qml.apply(op)

    return qml.expval(qml.Z(0))
    
resources = node()["resources"]
gate_counts = dict(resources.gate_types)
gate_counts

{'QubitUnitary': 4,
 'CNOT': 15,
 'RZ': 8,
 'RY': 6,
 'GlobalPhase': 1,
 'SelectPauliRot': 70,
 'RX': 1}

In [7]:
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(ros.asymmetric_two_qubit_decomp, show_matrices=False)(U, wires=wires))

In [8]:
from rotoptsynth.diag_decomps import _diag_decomp_two_qubits
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(_diag_decomp_two_qubits, show_matrices=False)(U, wires=wires))

0: ─╭U(M0)──RY(0.92)──RZ(1.63)─╭●──RX(1.31)─╭●──RZ(9.72)──RY(0.77)──RZ(1.15)─┤  
1: ─╰U(M0)──RY(1.14)──RZ(0.61)─╰X──RZ(0.81)─╰X──RZ(1.69)──RY(0.71)──RZ(3.99)─┤  


In [9]:
ros.attach_multiplexer_node([qml.RY(0.6, 0)], [qml.RY(-0.1, 0)], "mpx")

[SelectPauliRot(array([ 0.6, -0.1]), wires=['mpx', 0])]

In [10]:
ops0 = [qml.SelectPauliRot([0.6, 0.7], [0], target_wire="target", rot_axis="X")]
ops1 = [qml.SelectPauliRot([0.2, -0.5], [0], target_wire="target", rot_axis="X")]
ros.attach_multiplexer_node(ops0, ops1, "new mpx")

[SelectPauliRot(array([ 0.6,  0.7,  0.2, -0.5]), wires=['new mpx', 0, 'target'])]

In [11]:
ros.attach_multiplexer_node([qml.CNOT([0, 1])], [qml.CNOT([0, 1])], "mpx")

[CNOT(wires=[0, 1])]